# Stage 4 Explainer Notebook

This notebook is an interactive **study-first** guide for Stage 4.

It follows the 4-pass loop:
1. Problem framing
2. Intuition
3. Mechanics (shape-first)
4. Operatable practice and troubleshooting


## Prerequisites

- Run inside `red-book/src/stage-4` (or keep `STAGE_DIR` path below).
- Install dependencies from `requirements.txt` (and optional `requirements-gpu.txt`).
- Recommended: keep `STAGE4_PRESET=quick` while iterating.


## Recommended Run Order

Use one of these two paths:

### Fast Study Path (20-40 min)
1. `1) Preflight Checks`
2. `4) Explicit Tensor Shape Probes`
3. `5) Hardware Performance Benchmark (FP32 vs AMP)`
4. `7) Deliverable Automation`
5. `8) Inspect Results Quickly`

### Full Lab Path (90-180 min)
1. `1) Preflight Checks`
2. `2) Run Intermediate Topic Scripts`
3. `3) Optional: Stage Fail-Fast Runner + Output Verification`
4. `4) Explicit Tensor Shape Probes`
5. `5) Hardware Performance Benchmark (FP32 vs AMP)`
6. `6) Diagnostic Failure Labs`
7. `7) Deliverable Automation`
8. `8) Inspect Results Quickly`

Notes:
- Keep `STAGE4_PRESET=quick` during iteration; switch to `full` only for final evidence runs.
- If CUDA is unavailable, AMP comparison is skipped automatically.


In [ ]:
from __future__ import annotations

import csv
import json
import os
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import torch
except Exception:
    torch = None

STAGE = 4
CWD = Path.cwd()

if (CWD / f"run_all_stage{STAGE}.ps1").exists():
    STAGE_DIR = CWD
else:
    STAGE_DIR = Path(r"C:/Users/luixj/AI/red-book/src/stage-4").resolve()

RESULTS_DIR = STAGE_DIR / "results"
STAGE_RESULTS_DIR = RESULTS_DIR / f"stage{STAGE}"

print("Stage:", STAGE)
print("Stage dir:", STAGE_DIR)
print("Results dir:", RESULTS_DIR)
print("Stage results dir:", STAGE_RESULTS_DIR)


def run_cmd(cmd, cwd=STAGE_DIR):
    cmd_list = cmd if isinstance(cmd, list) else cmd.split()
    print("$", " ".join(map(str, cmd_list)))
    proc = subprocess.run(
        cmd_list,
        cwd=str(cwd),
        text=True,
        capture_output=True,
    )
    if proc.stdout:
        print(proc.stdout)
    if proc.stderr:
        print("[stderr]\n" + proc.stderr)
    if proc.returncode != 0:
        raise RuntimeError(f"Command failed (exit={proc.returncode}): {cmd_list}")
    return proc


def snapshot_results():
    if not RESULTS_DIR.exists():
        return {}
    snap = {}
    for p in RESULTS_DIR.rglob("*"):
        if p.is_file():
            snap[str(p.relative_to(RESULTS_DIR))] = p.stat().st_mtime
    return snap


def diff_results(before, after):
    new_files = sorted([name for name in after if name not in before])
    changed_files = sorted([
        name for name in after
        if name in before and after[name] != before[name]
    ])
    return new_files, changed_files


def run_script(script_name: str):
    before = snapshot_results()
    script_path = STAGE_DIR / script_name
    if not script_path.exists():
        raise FileNotFoundError(f"Missing script: {script_path}")
    run_cmd([sys.executable, script_name])
    after = snapshot_results()
    new_files, changed_files = diff_results(before, after)
    print("new files:", new_files)
    print("changed files:", changed_files)


def list_results(limit=80):
    if not RESULTS_DIR.exists():
        print("results directory does not exist yet")
        return
    files = sorted([p for p in RESULTS_DIR.rglob('*') if p.is_file()])
    print(f"total files: {len(files)}")
    for p in files[:limit]:
        rel = p.relative_to(RESULTS_DIR)
        print(f"- {rel} ({p.stat().st_size} bytes)")


def preview_csv(path: Path, rows: int = 5):
    if not path.exists():
        print("missing:", path)
        return
    with path.open('r', encoding='utf-8', newline='') as f:
        reader = csv.reader(f)
        for i, row in enumerate(reader):
            print(row)
            if i + 1 >= rows:
                break

os.environ.setdefault("STAGE4_PRESET", "quick")
print("STAGE4_PRESET:", os.environ.get("STAGE4_PRESET"))


## 1) Preflight Checks

In [ ]:
run_cmd([sys.executable, '--version'])

runner = STAGE_DIR / 'run_all_stage4.ps1'
verifier = STAGE_DIR / 'verify_stage4_outputs.ps1'
print('runner exists:', runner.exists(), runner)
print('verifier exists:', verifier.exists(), verifier)
print('stage scripts:', len(list(STAGE_DIR.glob('topic*.py'))))

if torch is not None:
    print('torch version:', torch.__version__)
    print('cuda available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('cuda device:', torch.cuda.get_device_name(0))


## 2) Run Intermediate Topic Scripts (Trace Middle Artifacts)

In [ ]:
topic_scripts = [
  "topic01_loop_anatomy.py",
  "topic02_mlp_intermediate.py",
  "topic03_cnn_intermediate.py",
  "topic04_rnn_intermediate.py",
  "topic05_transformer_intermediate.py",
  "topic07_failure_modes.py",
  "topic08_project_baseline.py"
]
print('topic scripts:', topic_scripts)
for script in topic_scripts:
    print(f"\n=== running {script} ===")
    run_script(script)


## 3) Optional: Stage Fail-Fast Runner + Output Verification

In [ ]:
run_cmd([
    'powershell', '-ExecutionPolicy', 'Bypass',
    '-File', f'run_all_stage{STAGE}.ps1',
    '-Preset', os.environ.get('STAGE4_PRESET', 'quick')
], cwd=STAGE_DIR)

verify_script = STAGE_DIR / f'verify_stage{STAGE}_outputs.ps1'
if verify_script.exists():
    run_cmd([
        'powershell', '-ExecutionPolicy', 'Bypass',
        '-File', verify_script.name
    ], cwd=STAGE_DIR)
else:
    print('verify script not found:', verify_script)


## 4) Explicit Tensor Shape Probes (Required)

Requirement:
- Check tensor shapes **after forward pass and before loss calculation**.
- Ensure model output batch dimension matches target batch dimension.


In [ ]:
# MLP shape probe
import torch
from sklearn.datasets import load_digits

raw = load_digits(as_frame=True).frame
feature_cols = [c for c in raw.columns if c != 'target']

x = torch.tensor((raw[feature_cols].to_numpy(dtype='float32') / 16.0), dtype=torch.float32)
y = torch.tensor(raw['target'].to_numpy(dtype='int64'), dtype=torch.long)

batch = 64
xb = x[:batch]
yb = y[:batch]

mlp = torch.nn.Sequential(
    torch.nn.Linear(64, 128),
    torch.nn.ReLU(),
    torch.nn.Linear(128, 10),
)

logits = mlp(xb)
print('MLP shape check')
print('input  :', tuple(xb.shape))
print('logits :', tuple(logits.shape))
print('target :', tuple(yb.shape))

assert logits.ndim == 2, 'Expected 2D logits [B, C]'
assert logits.shape[0] == yb.shape[0], 'Batch mismatch between logits and targets'
assert logits.shape[1] == 10, 'Expected 10 classes for digits'

loss_fn = torch.nn.CrossEntropyLoss()
loss = loss_fn(logits, yb)
print('loss OK:', float(loss.item()))


In [ ]:
# CNN shape probe
import torch
from sklearn.datasets import load_digits

raw = load_digits(as_frame=True).frame
feature_cols = [c for c in raw.columns if c != 'target']

x = torch.tensor((raw[feature_cols].to_numpy(dtype='float32') / 16.0), dtype=torch.float32)
x = x.view(-1, 1, 8, 8)  # NCHW
y = torch.tensor(raw['target'].to_numpy(dtype='int64'), dtype=torch.long)

batch = 64
xb = x[:batch]
yb = y[:batch]

cnn = torch.nn.Sequential(
    torch.nn.Conv2d(1, 16, kernel_size=3, padding=1),
    torch.nn.ReLU(),
    torch.nn.MaxPool2d(2),
    torch.nn.Conv2d(16, 32, kernel_size=3, padding=1),
    torch.nn.ReLU(),
    torch.nn.MaxPool2d(2),
    torch.nn.Flatten(),
    torch.nn.Linear(32 * 2 * 2, 10),
)

logits = cnn(xb)
print('CNN shape check')
print('input  :', tuple(xb.shape))
print('logits :', tuple(logits.shape))
print('target :', tuple(yb.shape))

assert logits.ndim == 2, 'Expected 2D logits [B, C]'
assert logits.shape[0] == yb.shape[0], 'Batch mismatch between logits and targets'
assert logits.shape[1] == 10, 'Expected 10 classes for digits'

loss_fn = torch.nn.CrossEntropyLoss()
loss = loss_fn(logits, yb)
print('loss OK:', float(loss.item()))


## 5) Hardware Performance Benchmark (FP32 vs AMP)

This cell benchmarks one training epoch and reports **images/sec**.
If CUDA is unavailable, it will run CPU-only and skip AMP comparison.


In [ ]:
import time
import torch
from sklearn.datasets import load_digits
from torch.utils.data import DataLoader, TensorDataset


def build_loader(batch_size=256):
    raw = load_digits(as_frame=True).frame
    feature_cols = [c for c in raw.columns if c != 'target']
    x = torch.tensor((raw[feature_cols].to_numpy(dtype='float32') / 16.0), dtype=torch.float32).view(-1, 1, 8, 8)
    y = torch.tensor(raw['target'].to_numpy(dtype='int64'), dtype=torch.long)
    ds = TensorDataset(x, y)
    return DataLoader(ds, batch_size=batch_size, shuffle=True, pin_memory=torch.cuda.is_available())


def make_model(device):
    return torch.nn.Sequential(
        torch.nn.Conv2d(1, 32, 3, padding=1),
        torch.nn.ReLU(),
        torch.nn.MaxPool2d(2),
        torch.nn.Conv2d(32, 64, 3, padding=1),
        torch.nn.ReLU(),
        torch.nn.MaxPool2d(2),
        torch.nn.Flatten(),
        torch.nn.Linear(64 * 2 * 2, 10),
    ).to(device)


def benchmark_epoch(use_amp: bool):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    loader = build_loader(batch_size=256)
    model = make_model(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
    loss_fn = torch.nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda' and use_amp))

    if device.type == 'cuda':
        torch.cuda.reset_peak_memory_stats(device)
        torch.backends.cudnn.benchmark = True
        torch.cuda.synchronize()

    seen = 0
    t0 = time.perf_counter()
    model.train()
    for xb, yb in loader:
        xb = xb.to(device, non_blocking=(device.type == 'cuda'))
        yb = yb.to(device, non_blocking=(device.type == 'cuda'))

        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type='cuda', enabled=(device.type == 'cuda' and use_amp)):
            logits = model(xb)
            loss = loss_fn(logits, yb)

        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        seen += xb.size(0)

    if device.type == 'cuda':
        torch.cuda.synchronize()

    elapsed = time.perf_counter() - t0
    ips = seen / elapsed
    peak_gb = (torch.cuda.max_memory_allocated(device) / 1e9) if device.type == 'cuda' else 0.0
    return {
        'device': device.type,
        'amp': use_amp,
        'samples': seen,
        'seconds': elapsed,
        'images_per_second': ips,
        'peak_vram_gb': peak_gb,
    }

base = benchmark_epoch(use_amp=False)
print('FP32:', base)

if torch.cuda.is_available():
    amp = benchmark_epoch(use_amp=True)
    print('AMP :', amp)
    speedup = amp['images_per_second'] / base['images_per_second']
    print(f"AMP speedup: {speedup:.3f}x")
else:
    print('CUDA not available: AMP comparison skipped in this environment.')


## 6) Diagnostic Failure Labs

Two controlled labs:
1. Vanishing gradients (deep Sigmoid stack)
2. Overfitting (tiny-train-set CNN)


In [ ]:
# Failure Lab A: Vanishing gradients (Sigmoid vs ReLU)
import matplotlib.pyplot as plt
import torch


def build_deep_mlp(activation='sigmoid', depth=8, width=64):
    layers = []
    in_dim = 64
    act_cls = torch.nn.Sigmoid if activation == 'sigmoid' else torch.nn.ReLU
    for _ in range(depth):
        layers.append(torch.nn.Linear(in_dim, width))
        layers.append(act_cls())
        in_dim = width
    layers.append(torch.nn.Linear(in_dim, 10))
    return torch.nn.Sequential(*layers)


def collect_grad_norms(model):
    x = torch.randn(256, 64)
    y = torch.randint(0, 10, (256,))
    loss_fn = torch.nn.CrossEntropyLoss()
    logits = model(x)
    loss = loss_fn(logits, y)
    loss.backward()

    norms = []
    names = []
    for name, p in model.named_parameters():
        if p.grad is not None and 'weight' in name:
            names.append(name)
            norms.append(float(p.grad.norm().item()))
    return names, norms

sig_model = build_deep_mlp('sigmoid', depth=10, width=64)
relu_model = build_deep_mlp('relu', depth=10, width=64)

sig_names, sig_norms = collect_grad_norms(sig_model)
_, relu_norms = collect_grad_norms(relu_model)

plt.figure(figsize=(10, 4))
plt.semilogy(sig_norms, marker='o', label='Sigmoid stack')
plt.semilogy(relu_norms, marker='o', label='ReLU stack')
plt.title('Gradient Norm by Layer (Vanishing Gradient Probe)')
plt.xlabel('Layer index (weight tensors in order)')
plt.ylabel('Gradient norm (log scale)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print('If Sigmoid curve collapses toward near-zero in early layers, vanishing gradient risk is present.')


In [ ]:
# Failure Lab B: Overfitting demo (tiny train subset CNN)
import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

raw = load_digits(as_frame=True).frame
feature_cols = [c for c in raw.columns if c != 'target']

x = torch.tensor((raw[feature_cols].to_numpy(dtype='float32') / 16.0), dtype=torch.float32).view(-1, 1, 8, 8)
y = torch.tensor(raw['target'].to_numpy(dtype='int64'), dtype=torch.long)

idx = np.arange(len(x))
train_idx, temp_idx = train_test_split(idx, test_size=0.7, random_state=404, stratify=y.numpy())
# Intentionally tiny train set to provoke overfitting
tiny_train_idx = train_idx[:120]
val_idx, _ = train_test_split(temp_idx, test_size=0.5, random_state=404, stratify=y.numpy()[temp_idx])

x_train, y_train = x[tiny_train_idx], y[tiny_train_idx]
x_val, y_val = x[val_idx], y[val_idx]

train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=32, shuffle=True)
val_loader = DataLoader(TensorDataset(x_val, y_val), batch_size=256, shuffle=False)

model = torch.nn.Sequential(
    torch.nn.Conv2d(1, 16, 3, padding=1),
    torch.nn.ReLU(),
    torch.nn.MaxPool2d(2),
    torch.nn.Conv2d(16, 32, 3, padding=1),
    torch.nn.ReLU(),
    torch.nn.MaxPool2d(2),
    torch.nn.Flatten(),
    torch.nn.Linear(32 * 2 * 2, 64),
    torch.nn.ReLU(),
    torch.nn.Linear(64, 10),
)

opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = torch.nn.CrossEntropyLoss()

train_loss_hist, val_loss_hist = [], []
train_acc_hist, val_acc_hist = [], []


def eval_epoch(loader):
    model.eval()
    total_loss, total_correct, total = 0.0, 0, 0
    with torch.no_grad():
        for xb, yb in loader:
            logits = model(xb)
            loss = loss_fn(logits, yb)
            total_loss += float(loss.item()) * xb.size(0)
            total_correct += int((logits.argmax(1) == yb).sum().item())
            total += xb.size(0)
    return total_loss / total, total_correct / total

for epoch in range(1, 61):
    model.train()
    for xb, yb in train_loader:
        logits = model(xb)
        loss = loss_fn(logits, yb)
        opt.zero_grad()
        loss.backward()
        opt.step()

    tr_loss, tr_acc = eval_epoch(train_loader)
    va_loss, va_acc = eval_epoch(val_loader)

    train_loss_hist.append(tr_loss)
    val_loss_hist.append(va_loss)
    train_acc_hist.append(tr_acc)
    val_acc_hist.append(va_acc)

    if epoch in (1, 10, 20, 40, 60):
        print(f"epoch={epoch:02d} train_loss={tr_loss:.4f} val_loss={va_loss:.4f} train_acc={tr_acc:.4f} val_acc={va_acc:.4f}")

plt.figure(figsize=(11,4))
plt.subplot(1,2,1)
plt.plot(train_loss_hist, label='train_loss')
plt.plot(val_loss_hist, label='val_loss')
plt.title('Overfitting Probe: Loss Curves')
plt.xlabel('epoch')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1,2,2)
plt.plot(train_acc_hist, label='train_acc')
plt.plot(val_acc_hist, label='val_acc')
plt.title('Overfitting Probe: Accuracy Curves')
plt.xlabel('epoch')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

final_gap = train_acc_hist[-1] - val_acc_hist[-1]
print(f"Final accuracy gap (train - val): {final_gap:.4f}")
if final_gap > 0.10:
    print('DIAGNOSIS: Overfitting pattern observed (memorization risk).')
else:
    print('DIAGNOSIS: No strong overfitting signal in this run.')


## 7) Deliverable Automation: Build `model_compare_before_after.csv`

In [ ]:
def build_model_compare_artifact(output_name='model_compare_before_after.csv'):
    stage_dir = STAGE_RESULTS_DIR if STAGE_RESULTS_DIR.exists() else RESULTS_DIR
    stage_dir.mkdir(parents=True, exist_ok=True)

    records = []
    for csv_path in stage_dir.rglob('*.csv'):
        try:
            df = pd.read_csv(csv_path)
        except Exception:
            continue

        # Preferred evidence schema
        if {'metric_name', 'before_value', 'after_value'}.issubset(df.columns):
            for _, row in df.iterrows():
                before = float(row.get('before_value'))
                after = float(row.get('after_value'))
                delta = float(row.get('delta')) if 'delta' in row and pd.notna(row.get('delta')) else after - before
                records.append({
                    'source_file': str(csv_path.relative_to(RESULTS_DIR)),
                    'metric_name': row.get('metric_name', 'unknown_metric'),
                    'before_value': before,
                    'after_value': after,
                    'delta': delta,
                    'decision': row.get('decision', 'n/a'),
                    'run_id': row.get('run_id', ''),
                    'topic_or_module': row.get('topic_or_module', ''),
                })

    out = pd.DataFrame(records)
    out_path = stage_dir / output_name
    if out.empty:
        print('No before/after schema rows found yet. Run stage scripts first.')
        out = pd.DataFrame(columns=[
            'source_file', 'metric_name', 'before_value', 'after_value',
            'delta', 'decision', 'run_id', 'topic_or_module'
        ])
    else:
        out = out.sort_values(['metric_name', 'delta'], ascending=[True, False])
    out.to_csv(out_path, index=False)
    print('Saved:', out_path)
    return out

model_compare_df = build_model_compare_artifact()
model_compare_df.head(20)


## 8) Inspect Results Quickly

In [ ]:
list_results(limit=120)

artifact = STAGE_RESULTS_DIR / 'model_compare_before_after.csv'
if artifact.exists():
    df = pd.read_csv(artifact)
    if not df.empty:
        summary = (
            df.assign(abs_delta=lambda d: d['delta'].abs())
              .sort_values(['abs_delta'], ascending=False)
              .drop(columns=['abs_delta'])
        )
        print('Top deltas:')
        display(summary.head(15))
    else:
        print('Artifact exists but is empty. Run more scripts/cells to populate it.')
else:
    print('Artifact not found:', artifact)
